# 1. Data processing

Converting the raw EEG recordings into topographic scalp maps for training.

The dataset is 45 recording sessions from 39 participants, each asked to rest, blink,
and make horizontal and vertical eye movements. Events are labelled and timestamped.

The functions used here are in `src/ocular`, so the same code runs from the command line.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from ocular import channels, epochs, manifest
from ocular.prepare import prepare
from ocular.topoplots import TopoplotRenderer

RAW = Path("../data/raw")

## Loading recordings

Each `.set` file is one session. The study folder and participant number form the group
key, which is the unit the train and test split is drawn over.

In [ ]:
recordings = epochs.find_recordings(RAW)
print(f"{len(recordings)} recordings")
for study, participant, path in recordings[:5]:
    print(f"  {study:12} {participant:8} {path.name}")

In [ ]:
example = epochs.normalise_events(epochs.load_recording(recordings[0][2]))
print(example)
print()
print("event codes:", example.event_id)

## Re-epoching around event onsets

The files are pre-epoched, but each epoch contains six to twenty events of a single type.
Training needs one event per example, so each epoch is cut into short windows centred on
the onsets marked in the eye tracker trigger channels.

This increases the number of usable examples by roughly a factor of ten.

In [ ]:
blinks = example[channels.EVENT_CODES["blink"]]

trigger = blinks.get_data(copy=False)[0, blinks.ch_names.index("eye-blink"), :]
fig, ax = plt.subplots(figsize=(10, 2.5))
ax.plot(blinks.times, trigger)
ax.set_title("Blink trigger channel, one parent epoch")
ax.set_xlabel("time (s)")
plt.show()

Onsets are the rising edges of the trigger channel. The channel is thresholded before
edge detection, since differencing it directly responds to noise on the baseline.

In [ ]:
windows = epochs.split_on_triggers(blinks, channels.TRIGGERS_BY_EVENT["blink"])
print(f"{len(blinks)} parent epochs became {len(windows)} single blink segments")
print(f"window length {windows[0].shape[1]} samples, "
      f"{epochs.TMIN}s to {epochs.TMAX}s around onset")

## Rest segments

Rest is a state rather than an event, so there is no trigger to align to. Resting epochs
are cut into fixed windows of the same length as the event windows.

Equal window lengths across classes prevent segment duration from acting as a cue, since
the next step averages each window over time.

In [ ]:
rest = epochs.split_rest(example[channels.EVENT_CODES["rest"]])
print(f"{len(rest)} rest segments, {rest[0].shape[1]} samples each")
assert rest[0].shape[1] == windows[0].shape[1]

## Channel selection

Three groups of channels are excluded before rendering.

Eye tracker trigger channels carry the event labels.

EOG electrodes sit beside the eyes and record the artifact almost directly. Excluding them
means the model classifies from scalp topography alone, so it can run on recordings with no
EOG montage. The ICA baseline in notebook 3 does use them.

Fp1 and Fp2 sit above the eyes and saturate on every blink.

In [ ]:
kept = channels.scalp_channels(example.ch_names)
dropped = [ch for ch in example.ch_names if ch not in kept]

print(f"kept {len(kept)}: {kept}")
print()
print(f"dropped {len(dropped)}: {dropped}")

## Topoplots

Each segment is averaged across its time window and rendered as a topoplot, a map of
electrical activity across the scalp. This reduces a multichannel time series to a single
image.

Blinks appear as strong symmetric frontal activity. Saccades are lateralised.

In [ ]:
renderer = TopoplotRenderer()
out = Path("../data/examples")

for event in ("blink", "rest", "h_saccade", "v_saccade"):
    segment = next(
        s for s in epochs.segments_for_recording(*recordings[0]) if s.event == event
    )
    renderer.render(segment, out / f"{event}.png")

fig, axes = plt.subplots(1, 4, figsize=(12, 3.5))
for ax, event in zip(axes, ("blink", "rest", "h_saccade", "v_saccade")):
    ax.imshow(plt.imread(out / f"{event}.png"))
    ax.set_title(event)
    ax.axis("off")
plt.show()

renderer.close()

## Rendering the full dataset

`prepare` applies the steps above to every recording and writes a manifest with one row
per image, recording its label, event type and source recording.

The command line equivalent is `ocular prepare`. It renders one matplotlib figure per
segment, so it takes a while.

In [ ]:
frame = prepare(RAW, Path("../data/topoplots"), Path("../data/manifest.csv"))
print(manifest.summarise(frame))

In [ ]:
frame.head()